# Cycle 1 — Tuning (Chronological Split)

The same RandomizedSearchCV setup from the original tuning notebook is reused here, but the outer train/test split is changed from random to chronological. Meaning the model is trained only on earlier seasons and tested on later seasons, giving a more realistic estimate of future prediction performance.

In [ ]:
import sys, os       
import pandas  as pd      
import numpy as np         
import warnings
warnings.filterwarnings('ignore')  

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
# RandomizedSearchCV: samples random param combinations instead of exhaustive grid
# StratifiedKFold: inner CV that preserves class ratios across folds
from sklearn.metrics import accuracy_score, classification_report  # evaluation metrics
from xgboost import XGBClassifier                  # XGBoost gradient boosting
from sklearn.ensemble import RandomForestClassifier  # ensemble of decision trees
from lightgbm import LGBMClassifier                # LightGBM gradient boosting


_here = os.getcwd()                                   
while not os.path.isdir(os.path.join(_here, 'data')): 
    _p = os.path.dirname(_here)                       
    if _p == _here: raise RuntimeError('project root not found')  
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)                           

from config import Paths, ensure_dirs  
ensure_dirs() 


## Prepare Data — Chronological Split

The dataset is sorted by Season in ascending order, and an 80/20 split is applied based on row position to create a strictly chronological train/test split. The Season column is then removed, as it is used only for ordering and not as a predictive feature.

In [2]:
# Sort chronologically by Season — ensures earlier rows = earlier seasons
df1 = pd.read_csv(str(Paths.PL_MATCHES_PROCESSED)).sort_values('Season').reset_index(drop=True)
split_idx1 = int(len(df1) * 0.8)          # cut at 80% — same split ratio as random tuning

# Drop FTR (target) and Season (used only for sorting, not a feature)
X1_train = df1.iloc[:split_idx1].drop(columns=['FTR', 'Season'])  # first 80% of seasons = training
y1_train = df1.iloc[:split_idx1]['FTR']                           # training labels
X1_test  = df1.iloc[split_idx1:].drop(columns=['FTR', 'Season'])  # last 20% of seasons = test
y1_test  = df1.iloc[split_idx1:]['FTR']                           # test labels

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # inner CV for the search (training set only)

print(f'Train: {len(X1_train)} matches | Test: {len(X1_test)} matches')
print(f'Features ({X1_train.shape[1]}): {list(X1_train.columns)}')


Train: 5472 matches | Test: 1368 matches
Features (33): ['HomeTeam', 'AwayTeam', 'HTGS', 'ATGS', 'HTGC', 'ATGC', 'HTP', 'ATP', 'HM1', 'HM2', 'HM3', 'HM4', 'HM5', 'AM1', 'AM2', 'AM3', 'AM4', 'AM5', 'MW', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts']


## XGBoost Hyperparameter Search (Chronological)

Runs `RandomizedSearchCV` with 50 combinations over a 7-parameter XGBoost grid, evaluated via 5-fold stratified CV on the chronological training set.

In [3]:
xgb_param_grid = {
    'n_estimators':     [100,200,300,500],    # number of boosting rounds
    'max_depth':        [3,4,5,6],             # tree depth — controls bias/variance tradeoff
    'learning_rate':    [0.01,0.05,0.1,0.2],  # shrinkage factor per tree
    'subsample':        [0.7,0.8,1.0],         # row subsampling fraction
    'colsample_bytree': [0.7,0.8,1.0],         # feature subsampling fraction
    'min_child_weight': [1,3,5],               # min sum of instance weights per leaf
    'gamma':            [0,0.1,0.2],           # min loss reduction for a split
}

xgb1 = XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)  # base model
search_d1 = RandomizedSearchCV(
    xgb1, xgb_param_grid,
    n_iter=50,          # try 50 random combinations
    cv=cv,              # 5-fold stratified CV on training set
    scoring='accuracy', # optimise for accuracy
    random_state=42,    # reproducible sampling
    n_jobs=-1,          # parallelise across all CPU cores
    verbose=1           # print progress
)
search_d1.fit(X1_train, y1_train)  # run the search

print('Best params:', search_d1.best_params_)
print(f'Best CV accuracy: {search_d1.best_score_*100:.2f}%')   # best average accuracy across 5 folds
y_pred_xgb_d1 = search_d1.best_estimator_.predict(X1_test)     # predict with best model on held-out test
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_xgb_d1)*100:.2f}%')  # final test accuracy
print()
print(classification_report(y1_test, y_pred_xgb_d1, target_names=['Away Win','Draw','Home Win']))


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 0.8, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.7}
Best CV accuracy: 53.13%
Test accuracy:    52.05%

              precision    recall  f1-score   support

    Away Win       0.52      0.42      0.46       406
        Draw       0.35      0.06      0.10       348
    Home Win       0.53      0.85      0.65       614

    accuracy                           0.52      1368
   macro avg       0.47      0.44      0.41      1368
weighted avg       0.48      0.52      0.46      1368



## Random Forest Hyperparameter Search (Chronological)

Runs `RandomizedSearchCV` over 7 Random Forest parameters using the same 5-fold CV on the chronological training set.

In [4]:
rf_param_grid = {
    'n_estimators':      [100, 200, 300, 500],  # number of trees in the forest
    'max_depth':         [10, 15, 20, 25],       # max depth per tree — deeper = more complex
    'min_samples_split': [2, 5, 10],             # min samples to split a node — higher = regularisation
    'min_samples_leaf':  [1, 2, 4],              # min samples in a leaf node
    'max_features':      ['sqrt', 'log2'],        # features considered per split: sqrt = classic RF choice
    'bootstrap':         [True, False],           # True = bagging rows; False = use full dataset per tree
    'class_weight':      ['balanced', 'balanced_subsample']  # handle class imbalance
}

rf1 = RandomForestClassifier(random_state=42)  # base RF model
search_rf_d1 = RandomizedSearchCV(
    rf1, rf_param_grid,
    n_iter=50,          # 50 random combinations
    cv=cv,              # same 5-fold CV
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_rf_d1.fit(X1_train, y1_train)  # run search on training set

print('Best params (Random Forest):', search_rf_d1.best_params_)
print(f'Best CV accuracy: {search_rf_d1.best_score_*100:.2f}%')
y_pred_rf_d1 = search_rf_d1.best_estimator_.predict(X1_test)  # predict with best RF model
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_rf_d1)*100:.2f}%')
print()
print(classification_report(y1_test, y_pred_rf_d1, target_names=['Away Win','Draw','Home Win']))


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params (Random Forest): {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20, 'class_weight': 'balanced_subsample', 'bootstrap': True}
Best CV accuracy: 51.26%
Test accuracy:    51.02%

              precision    recall  f1-score   support

    Away Win       0.48      0.46      0.47       406
        Draw       0.28      0.12      0.17       348
    Home Win       0.57      0.76      0.65       614

    accuracy                           0.51      1368
   macro avg       0.44      0.45      0.43      1368
weighted avg       0.47      0.51      0.48      1368



## LightGBM Hyperparameter Search (Chronological)

Runs `RandomizedSearchCV` over 8 LightGBM parameters with 50 iterations and 5-fold CV on the chronological training set.

In [5]:
lgb_param_grid = {
    'n_estimators':      [100, 200, 300, 500],   # boosting rounds
    'learning_rate':     [0.01, 0.05, 0.1, 0.15],  # shrinkage per tree
    'max_depth':         [3, 4, 5, 6, 7],         # max depth
    'num_leaves':        [20, 31, 50, 100],        # main LightGBM complexity control
    'subsample':         [0.7, 0.8, 0.9, 1.0],    # row subsampling
    'colsample_bytree':  [0.7, 0.8, 0.9, 1.0],    # feature subsampling
    'min_child_samples': [10, 20, 30],             # min samples per leaf — higher = regularisation
    'reg_lambda':        [0, 0.1, 1.0]            # L2 regularisation on leaf weights
}

lgb1 = LGBMClassifier(random_state=42, verbose=-1)  # base LightGBM model
search_lgb_d1 = RandomizedSearchCV(
    lgb1, lgb_param_grid,
    n_iter=50,          # 50 random combinations
    cv=cv,              # same 5-fold CV
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_lgb_d1.fit(X1_train, y1_train)  # run search

print('Best params (LightGBM):', search_lgb_d1.best_params_)
print(f'Best CV accuracy: {search_lgb_d1.best_score_*100:.2f}%')
y_pred_lgb_d1 = search_lgb_d1.best_estimator_.predict(X1_test)  # predict with best LightGBM model
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_lgb_d1)*100:.2f}%')
print()
print(classification_report(y1_test, y_pred_lgb_d1, target_names=['Away Win','Draw','Home Win']))


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params (LightGBM): {'subsample': 0.8, 'reg_lambda': 1.0, 'num_leaves': 100, 'n_estimators': 500, 'min_child_samples': 10, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
Best CV accuracy: 53.25%
Test accuracy:    52.05%

              precision    recall  f1-score   support

    Away Win       0.53      0.43      0.48       406
        Draw       0.26      0.05      0.08       348
    Home Win       0.53      0.85      0.65       614

    accuracy                           0.52      1368
   macro avg       0.44      0.44      0.40      1368
weighted avg       0.46      0.52      0.46      1368



## Side-by-side comparison: Chronological vs Random tuning

### Random-split tuning results:

- Tunned XGboots accuracy: $53.65 \%$

- Tunned LightGBM accuracy: $53.29\%$

### Chronological-split tuning results:

 - Tunned XGboots accuracy: $52.05 \%$

 - Tunned LightGBM accuracy: $52.05 \%$

We can notice that the tunned models on chronological split are performing lower than the ones on random split, but eventhough chronological split show accuracy decreasing with roughly $1 \%$ it still the best choice to use since we are dealing with time-series data. 

- Based on the chronological results, XGBoost and LightGBM have the same test accuracy, but on Cross-Validation LightGBM is slightly better, marginal difference (53.25 for lightGBM vs 53.13 for XGBoost), so we are gonna choose LightGBM for the final model.


## Save the deployed Cycle 1 model

This notebook produces the **deployed** Cycle 1 model — chronological tuning is the honest evaluation, so its winner is what the API serves.


In [6]:
import joblib  # serialise trained model to disk

best_lgb = search_lgb_d1.best_estimator_ 
print(best_lgb)

joblib.dump(best_lgb,               str(Paths.C1_MODEL))     # save model to models/cycle1/cycle1_lgb_best.pkl
joblib.dump(list(X1_train.columns), str(Paths.C1_FEATURES));  # save feature column list — needed by the API


LGBMClassifier(colsample_bytree=0.7, learning_rate=0.01, max_depth=3,
               min_child_samples=10, n_estimators=500, num_leaves=100,
               random_state=42, reg_lambda=1.0, subsample=0.8, verbose=-1)
